# Shift-Scan — Executable Real-Data Experiments
### FineWeb-Edu validation, Mamba-2 130M, decay quantization, ternary projections, and pilot-result analysis

This notebook is designed to be run in **Google Colab with a GPU**. Unlike the earlier planning notebook, it contains runnable experiment code that loads the uploaded Shift-Scan source ZIP, fetches real text data, evaluates model variants, and writes JSON/CSV results and plots.

**What it actually measures**
1. Run the repository's correctness tests.
2. Fetch a reproducible sample from FineWeb-Edu and tokenize it.
3. Evaluate pretrained Mamba-2 130M with:
   - full-precision exponential decay / projections (reference),
   - shift-decay with full-precision projections,
   - shift-decay with ternary projections (initial quantization, before QAT).
4. Collect real model-derived decay values and compare integer-k quantization grids and horizon error.
5. Analyze the archived Seed-2 pilot results from the ZIP (clearly marked as prior results, not newly reproduced).
6. Save measurements and plots under `results/`.

**Research caution:** The shift+ternary result here is *pre-QAT* unless you separately run distillation. Do not present it as the trained student result. GPU timing is not included unless you run the optional benchmark section. Data sample sizes are a quick experiment; increase them for paper-quality results.

## 1. Runtime setup (Colab)

Set Colab to a GPU runtime first: **Runtime → Change runtime type → GPU**. This cell installs the project dependencies. It may take a few minutes.

In [ ]:
!nvidia-smi -L
!pip -q install datasets transformers huggingface_hub tqdm triton

## 2. Upload and extract your actual Shift-Scan ZIP

Upload the ZIP archive you supplied in this chat (the `shift_scan` project archive). The code selects the uploaded ZIP, extracts it, and locates `ssm_core.py`.

In [ ]:
from google.colab import files
import zipfile, os, glob, pathlib, shutil

uploaded = files.upload()
zip_candidates = [name for name in uploaded if name.lower().endswith(".zip")]
assert zip_candidates, "Please upload the Shift-Scan project ZIP."
ZIP_PATH = zip_candidates[0]

EXTRACT_ROOT = pathlib.Path("/content/shift_scan_project")
if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)
EXTRACT_ROOT.mkdir(parents=True)

with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall(EXTRACT_ROOT)

core_candidates = list(EXTRACT_ROOT.rglob("ssm_core.py"))
assert core_candidates, "ssm_core.py was not found in the ZIP. Check the uploaded archive."
PROJECT_DIR = core_candidates[0].parent
os.chdir(PROJECT_DIR)
print("Project directory:", PROJECT_DIR)
print("Files:", sorted(p.name for p in PROJECT_DIR.iterdir()))

## 3. Verify environment and run repository correctness tests

The tests are CPU-compatible for their core checks, though the optional official Mamba comparison needs CUDA and `mamba_ssm`. If a test fails, stop and resolve it before interpreting research metrics.

In [ ]:
import torch, sys, subprocess, json, platform, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 2
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

test_proc = subprocess.run([sys.executable, "tests.py"], text=True, capture_output=True)
print(test_proc.stdout)
if test_proc.stderr:
    print(test_proc.stderr[-4000:])
assert test_proc.returncode == 0, "Repository tests failed; do not continue to model evaluation." 

## 4. Fetch real FineWeb-Edu text and prepare a held-out token sample

We use a fixed stream order and reserve a small sample for evaluation. This is a quick validation experiment—not the same evaluation split or token budget as the archived training pilot. The data pipeline tokenizes actual text with the project's tokenizer.

In [ ]:
from datasets import load_dataset
from ssm_core import get_tokenizer

DATASET_NAME = "HuggingFaceFW/fineweb-edu"
DATASET_CONFIG = "sample-10BT"
SEQ_LEN = 512
N_DOCS = 24  # quick sample; increase (e.g. 128 or 512) for more stable estimates

tokenizer = get_tokenizer()
stream = load_dataset(DATASET_NAME, name=DATASET_CONFIG, split="train", streaming=True)
doc_iter = iter(stream)
token_chunks = []
docs_used = 0
for _ in range(N_DOCS):
    try:
        example = next(doc_iter)
    except StopIteration:
        break
    text = example.get("text", "")
    ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    if ids:
        token_chunks.extend(ids)
        token_chunks.append(tokenizer.eos_token_id)
        docs_used += 1

# Form contiguous fixed-length examples. Each sequence has input and next-token target.
tokens = torch.tensor(token_chunks, dtype=torch.long)
n_sequences = len(tokens) // (SEQ_LEN + 1)
assert n_sequences >= 2, f"Only {len(tokens)} tokens fetched; increase N_DOCS."
tokens = tokens[:n_sequences * (SEQ_LEN + 1)].view(n_sequences, SEQ_LEN + 1)
X = tokens[:, :-1].contiguous()
Y = tokens[:, 1:].contiguous()
print(f"Dataset: {DATASET_NAME}/{DATASET_CONFIG}")
print(f"Documents used: {docs_used} | total tokens: {len(token_chunks):,}")
print(f"Evaluation sequences: {len(X)} × {SEQ_LEN} tokens")
print("Example text:", tokenizer.decode(tokens[0, :80].tolist()))

## 5. Load Mamba-2 130M and evaluate variants on the same FineWeb sample

Each model loads the same pretrained checkpoint. The evaluation reports token-average cross-entropy and perplexity. It intentionally evaluates the quantized student **before QAT**, to isolate initial quantization damage.

Models are evaluated sequentially to reduce GPU memory use. If your runtime runs out of memory, reduce `SEQ_LEN`, `N_EVAL_SEQS`, or use a larger GPU.

In [ ]:
from ssm_core import QuantConfig, load_pretrained
import math, gc

N_EVAL_SEQS = min(8, len(X))  # adjust upward for more reliable estimates
EVAL_BATCH = 1
AMP_DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

def eval_model_on_tokens(model, x, y, batch_size=1):
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    with torch.no_grad():
        for start in range(0, len(x), batch_size):
            xb = x[start:start+batch_size].to(DEVICE)
            yb = y[start:start+batch_size].to(DEVICE)
            if DEVICE == "cuda":
                with torch.autocast("cuda", dtype=AMP_DTYPE):
                    logits = model(xb)
            else:
                logits = model(xb)
            loss_sum = torch.nn.functional.cross_entropy(
                logits.float().reshape(-1, logits.shape[-1]),
                yb.reshape(-1), reduction="sum"
            )
            total_loss += loss_sum.item()
            total_tokens += yb.numel()
            del logits, xb, yb
    mean_ce = total_loss / total_tokens
    return {"cross_entropy": mean_ce, "perplexity": float(math.exp(min(mean_ce, 30))),
            "tokens_evaluated": total_tokens}

eval_X, eval_Y = X[:N_EVAL_SEQS], Y[:N_EVAL_SEQS]
experiment_rows = []

variant_specs = [
    ("FP exp reference", QuantConfig(decay="exp", ternary=False)),
    ("Shift decay / FP projections", QuantConfig(decay="shift", ternary=False)),
    ("Shift decay / ternary projections (pre-QAT)", QuantConfig(decay="shift", ternary=True, group_size=128)),
]

for label, qcfg in variant_specs:
    print("\nLoading:", label, "| config:", qcfg)
    model, mcfg = load_pretrained("state-spaces/mamba2-130m", qcfg)
    model = model.to(DEVICE)
    if DEVICE == "cuda":
        model = model.half()
    metrics = eval_model_on_tokens(model, eval_X, eval_Y, EVAL_BATCH)
    row = {"variant": label, **metrics}
    experiment_rows.append(row)
    print(row)
    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

eval_df = pd.DataFrame(experiment_rows)
display(eval_df)
eval_df.to_csv("results_fineweb_model_variants.csv", index=False)
eval_df.to_json("results_fineweb_model_variants.json", orient="records", indent=2)
print("Saved results_fineweb_model_variants.csv and .json")

## 6. Real model-derived decay sensitivity and horizon error

This experiment loads the reference checkpoint with shift decay, performs a forward pass on the FineWeb sequences, and collects the model's actual continuous `k` values per layer/head. We compare quantization grids (step 1.0 vs 0.5) and estimate decay coefficient error over horizons.

For this implementation the decay coefficient is \(a(k)=1-2^{-k}\). This cell evaluates the approximation error on `k` values derived from the pretrained model and the real text sample; it does not train a codebook.

In [ ]:
from ssm_core import Mamba2Mixer

qcfg_shift = QuantConfig(decay="shift", ternary=False, k_step=1.0)
decay_model, mcfg = load_pretrained("state-spaces/mamba2-130m", qcfg_shift)
decay_model = decay_model.to(DEVICE).eval()
if DEVICE == "cuda":
    decay_model = decay_model.half()

decay_model.set_track_k(True)
with torch.no_grad():
    xb = eval_X[:min(2, len(eval_X))].to(DEVICE)
    if DEVICE == "cuda":
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            _ = decay_model(xb)
    else:
        _ = decay_model(xb)

k_values = []
for layer in decay_model.backbone.layers:
    mixer = layer.mixer
    if mixer.last_k is not None:
        k_values.append(mixer.last_k.detach().float().flatten().cpu())
k_cont = torch.cat(k_values).numpy()
k_cont = k_cont[np.isfinite(k_cont)]
assert len(k_cont) > 0
print("Collected continuous k values:", len(k_cont))
print(pd.Series(k_cont).describe())

def decay_grid_metrics(k_cont, step, horizons=(1, 8, 32, 128, 512, 2048)):
    k_q = np.clip(np.round(k_cont / step) * step, 1.0, 14.0)
    a_ref = 1.0 - np.exp2(-k_cont)
    a_q = 1.0 - np.exp2(-k_q)
    rows = []
    for horizon in horizons:
        # absolute discrepancy of accumulated decay response after horizon steps
        response_ref = np.power(a_ref, horizon)
        response_q = np.power(a_q, horizon)
        rows.append({
            "k_grid_step": step,
            "horizon": horizon,
            "mean_abs_decay_coeff_error": float(np.mean(np.abs(a_ref-a_q))),
            "mean_abs_horizon_response_error": float(np.mean(np.abs(response_ref-response_q))),
            "p95_abs_horizon_response_error": float(np.quantile(np.abs(response_ref-response_q), 0.95)),
        })
    return rows

decay_rows = decay_grid_metrics(k_cont, 1.0) + decay_grid_metrics(k_cont, 0.5)
decay_df = pd.DataFrame(decay_rows)
display(decay_df)
decay_df.to_csv("results_decay_horizon_error.csv", index=False)
decay_df.to_json("results_decay_horizon_error.json", orient="records", indent=2)

plt.figure(figsize=(8, 4.5))
for step in sorted(decay_df["k_grid_step"].unique()):
    subset = decay_df[decay_df["k_grid_step"] == step]
    plt.plot(subset["horizon"], subset["mean_abs_horizon_response_error"], marker="o", label=f"k grid step={step}")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("Horizon (steps)")
plt.ylabel("Mean absolute response error")
plt.title("Decay quantization error using model-derived k values")
plt.grid(True, alpha=0.25)
plt.legend()
plt.show()

del decay_model
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

## 7. Analyze the archive's existing Seed-2 pilot measurements

These are previously recorded results in `seed2_results_summary.json`, not freshly reproduced in this notebook. The archive documents nearest, horizon-weighted (HARPQ), and two-term dyadic variants, including completeness notes. This section visualizes available FineWeb validation perplexity and WikiText word perplexity while retaining the caveat that two variants' data are partial/rounded.

In [ ]:
summary_path = PROJECT_DIR / "seed2_results_summary.json"
assert summary_path.exists(), "seed2_results_summary.json not found in the uploaded project."
pilot = json.loads(summary_path.read_text())

pilot_rows = []
for run_name, run in pilot["runs"].items():
    train_eval = run.get("train_eval_step_305", {})
    downstream = run.get("downstream", {})
    wikitext = downstream.get("wikitext", {})
    pilot_rows.append({
        "run": run_name,
        "variant": run.get("variant"),
        "fineweb_student_ppl": train_eval.get("student_ppl"),
        "wikitext_word_ppl": wikitext.get("word_perplexity"),
        "lambada_acc": downstream.get("lambada_openai", {}).get("acc"),
        "data_completeness": run.get("data_source", ""),
    })

pilot_df = pd.DataFrame(pilot_rows)
display(pilot_df[["run", "variant", "fineweb_student_ppl", "wikitext_word_ppl", "lambada_acc"]])
pilot_df.to_csv("archived_seed2_pilot_results.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric, title in [
    (axes[0], "fineweb_student_ppl", "Archived FineWeb validation PPL"),
    (axes[1], "wikitext_word_ppl", "Archived WikiText word PPL"),
]:
    plot_df = pilot_df.dropna(subset=[metric])
    ax.bar(plot_df["variant"], plot_df[metric])
    ax.set_title(title)
    ax.set_ylabel(metric)
    ax.tick_params(axis="x", rotation=20)
    ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

print("Archived pilot description:", pilot.get("description"))
print("Archive notes:")
for note in pilot.get("notes", []):
    print("-", note)

## 8. Save a reproducibility manifest for this notebook run

The manifest captures environment, seed, code revision when available, evaluation sample metadata, and variant results. A git commit is recorded if this extracted project has a `.git` directory; ZIP snapshots often do not, so the notebook also hashes key source files.

In [ ]:
import hashlib, subprocess, datetime

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

source_hashes = {}
for filename in ["ssm_core.py", "train_qat.py", "kernels.py", "tests.py"]:
    fpath = PROJECT_DIR / filename
    if fpath.exists():
        source_hashes[filename] = sha256_file(fpath)

try:
    git_commit = subprocess.check_output(["git", "-C", str(PROJECT_DIR), "rev-parse", "HEAD"],
                                        text=True, stderr=subprocess.DEVNULL).strip()
except Exception:
    git_commit = None

manifest = {
    "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "seed": SEED,
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "device": DEVICE,
    "gpu_name": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
    "git_commit": git_commit,
    "source_sha256": source_hashes,
    "dataset": {"name": DATASET_NAME, "config": DATASET_CONFIG, "docs_used": docs_used,
                "eval_sequences": int(N_EVAL_SEQS), "sequence_length": int(SEQ_LEN)},
    "model": "state-spaces/mamba2-130m",
    "eval_variants": experiment_rows,
    "decay_grid_metrics": decay_rows,
    "notes": "Quick-run FineWeb sample; shift+ternary variant is evaluated before QAT."
}
pathlib.Path("experiment_manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2)[:6000])
print("Saved experiment_manifest.json")

## 9. Optional: run QAT distillation (long-running training)

The above cells produce actual model/data experiments without launching a lengthy training run. For a trained student, run the project's existing `train_qat.py` after correctness tests pass. Start with a short pilot (e.g. 2M tokens), then use a larger budget (10M–100M+) and multiple seeds for research-grade conclusions.

**Caution:** training resource requirements vary by GPU. The example below is commented out intentionally; uncomment and adjust only after confirming the Colab GPU and runtime budget.

In [ ]:
# Example short QAT pilot (uncomment to run):
# !python train_qat.py --name notebook_qat_pilot --decay shift --ternary 1 \
#     --tokens 2e6 --seq-len 512 --micro-bs 1 --accum 4 --dtype fp16 \
#     --val-seqs 8 --eval-every 50 --save-every 50 --log-every 10 --out runs

## 10. Download experiment outputs

CSV/JSON result files and the manifest are saved in the current project directory. Download them to retain raw data and keep your experiments auditable.

In [ ]:
output_files = [
    "results_fineweb_model_variants.csv",
    "results_fineweb_model_variants.json",
    "results_decay_horizon_error.csv",
    "results_decay_horizon_error.json",
    "archived_seed2_pilot_results.csv",
    "experiment_manifest.json",
]
print("Files created:")
for filename in output_files:
    if pathlib.Path(filename).exists():
        print(f"  {filename}: {pathlib.Path(filename).stat().st_size:,} bytes")

# In Colab, uncomment to download one-by-one:
# from google.colab import files
# for filename in output_files:
#     if pathlib.Path(filename).exists():
#         files.download(filename)